<a href="https://colab.research.google.com/github/mirian2004/AI-AI-/blob/Day13/Day13_Level2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**패키지 설치**

In [2]:
!pip install -q -U langchain-google-genai langchain-community langchain-core langchain-text-splitters faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


**라이브러리 불러오기 + API 설정**

In [3]:
import os
import google.generativeai as genai
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from google.colab import userdata

# 1. API 설정
# ⚠️ 'gemini_api_key' 부분은 왼쪽 보안 비밀(열쇠 아이콘) 패널에 등록한
#    실제 이름과 정확히 똑같이 맞춰주세요 (대소문자, 언더스코어/하이픈 포함)
GENAI_API_KEY = userdata.get('gemini_api_key')

genai.configure(api_key=GENAI_API_KEY)

print("API 설정 완료!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_1025/1316533612.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


API 설정 완료!


**예시 문서 텍스트**

In [4]:
long_text = """
1. Gemini 모델 개요
Gemini는 구글에서 개발한 차세대 멀티모달 AI 모델입니다. 텍스트뿐만 아니라 이미지, 오디오, 비디오, 코드를 이해하고 처리할 수 있는 능력을 갖추고 있습니다.
가장 큰 특징은 방대한 컨텍스트 창을 지원하여 수천 페이지의 문서를 한 번에 이해할 수 있다는 점입니다.

2. RAG(검색 증강 생성)의 중요성
AI 모델은 학습된 시점 이후의 최신 정보를 알지 못하는 '지식 컷오프' 현상이 발생합니다.
이를 해결하기 위해 RAG 기술을 사용합니다. RAG는 모델 외부에 벡터 데이터베이스를 구축하고, 질문과 관련된 문서를 실시간으로 검색하여 AI에게 전달합니다.
이 과정은 크게 로드(Load), 분할(Split), 임베딩(Embed), 저장(Store), 검색(Retrieve), 생성(Generate) 단계로 나뉩니다.

3. FAISS 데이터베이스란?
FAISS(Facebook AI Similarity Search)는 대규모 벡터 집합에서 유사한 벡터를 빠르게 찾기 위한 라이브러리입니다.
고차원 데이터를 처리할 때 매우 효율적이며, RAM 메모리 기반으로 동작하여 속도가 매우 빠릅니다.

4. 프로젝트 실습 주의사항
실습 시에는 반드시 구글 AI 스튜디오에서 발급받은 API 키를 보안에 유의하여 관리해야 합니다.
또한 무료 티어 사용 시 분당 호출 횟수(RPM) 제한이 있으므로 너무 짧은 간격으로 요청을 보내지 않도록 주의해야 합니다.
"""

**문서 분할 (Chunking)**

In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = text_splitter.create_documents([long_text])

print(f"문서 조각 개수: {len(docs)}개")

문서 조각 개수: 5개


**벡터 DB 생성 (Embedding + FAISS)**

**Gemini API 대신 HuggingFace의 로컬 임베딩 모델을 다운로드해서 사용**

Gemini API 임베딩 : 구글 서버에서 실행, API 키 필요, 네트워크 왕복 시간 있음.
HuggingFace : 내 Colab 세션 안에서 실행, API 키 불필요, 한번 모델 로드되면 빠름(특히 문서 개수 많을때.)

문서가 많아지면 Gemini API로 임베딩할 때마다 요청이 나가면서 무료 티어 RPM에 제한이 걸리기 쉽지만, 로컬 모델은 이 제한이 없음.
Level.2는 벡터 DB 구축은 로컬에서 즉시 처리되고, 답변 생성에만 Gemini API를 사용하는 구조. 즉 검색용 임베딩은 로컬, 답변 생성은 Gemini로 역할을 나눔.

In [6]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

print("벡터 DB 구축 완료!")

/tmp/ipykernel_1025/112945654.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

벡터 DB 구축 완료!


**Q&A 파이프라인 함수**

In [7]:
def ask_question(query):
    related_docs = vectorstore.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in related_docs])

    model = genai.GenerativeModel('gemini-flash-latest')

    prompt = f"""아래의 참고 정보를 바탕으로 질문에 답하세요.
    정보에 없는 내용은 모른다고 답하세요.

    [참고 정보]
    {context}

    질문: {query}"""

    response = model.generate_content(prompt)
    return response.text, context

**실습 테스트**

In [10]:
question = "FAISS의 특징이 뭐야?"
answer, retrieved_context = ask_question(question)

print(f"\n[질문]: {question}")
print(f"\n[검색된 문서 조각]:\n{retrieved_context}")
print(f"\n[AI 답변]:\n{answer}")


[질문]: FAISS의 특징이 뭐야?

[검색된 문서 조각]:
3. FAISS 데이터베이스란?
FAISS(Facebook AI Similarity Search)는 대규모 벡터 집합에서 유사한 벡터를 빠르게 찾기 위한 라이브러리입니다.
고차원 데이터를 처리할 때 매우 효율적이며, RAM 메모리 기반으로 동작하여 속도가 매우 빠릅니다.
4. 프로젝트 실습 주의사항
실습 시에는 반드시 구글 AI 스튜디오에서 발급받은 API 키를 보안에 유의하여 관리해야 합니다.
또한 무료 티어 사용 시 분당 호출 횟수(RPM) 제한이 있으므로 너무 짧은 간격으로 요청을 보내지 않도록 주의해야 합니다.

[AI 답변]:
제공된 정보에 따른 FAISS의 특징은 다음과 같습니다.

1. **대규모 벡터 집합에서 유사한 벡터를 빠르게 찾기 위한 라이브러리**입니다.
2. **고차원 데이터를 처리할 때 매우 효율적**입니다.
3. **RAM 메모리 기반으로 동작**하여 속도가 매우 빠릅니다.
